# Week 6 with new API interface

In [1]:
import jax
jax.config.update("jax_enable_x64", True) 

In [2]:
from src.api.v5 import GaussEmission, StaticTransition, Params
from drivers.utils import load_y_df
import numpy as np
import jax.numpy as jnp
from dataclasses import dataclass

ROOM_SLUG, TAG  = "room_001", "30min"  
ys_train_df = load_y_df(ROOM_SLUG, TAG, "train")  # the driver's X_*.csv are not used: 
ys_train_df = ys_train_df.set_index("datetime").sort_index()  # sort by datetime, just in case 

print(f"Loaded {ROOM_SLUG}/{TAG} — {ys_train_df.shape[0]} training observations") 
print(f"First 5 rows of training data:\n{ys_train_df.head()}")

Loaded room_001/30min — 2752 training observations
First 5 rows of training data:
                            co2  humidity
datetime                                 
2023-11-13 04:00:00  502.786667     48.05
2023-11-13 04:30:00  472.413333     48.05
2023-11-13 05:00:00  493.706667     48.05
2023-11-13 05:30:00  498.000000     48.05
2023-11-13 06:00:00  481.293333     47.92


In [4]:
@dataclass(frozen=True)
class Data:
    ys: jnp.ndarray
    datetime: np.ndarray
    ys_co2: jnp.ndarray
    ys_humidity: jnp.ndarray


traindata = Data(
    ys=jnp.array(ys_train_df.values),
    datetime=ys_train_df.index.to_numpy(),
    ys_co2=jnp.array(ys_train_df["co2"].values),
    ys_humidity=jnp.array(ys_train_df["humidity"].values),
)

### Fitting Univariate Model